# Part 9, notebook 1 of 5: where a person belongs in the pipeline

Companion to *Human in the Loop*. The five notebooks follow the post in order. Each explains the idea with a daily-life picture, shows the real source, and runs it. **Cells marked "Your turn" are for you to change and rerun.**

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 09a | `09a_where_a_human_belongs` | Where a person belongs in the pipeline | no |
| 09b | `09b_pause_and_resume` | Pause, ask, resume: `interrupt()` | no |
| 09c | `09c_who_to_ask` | Which cases to send: the review budget | no (reads the saved Part 8 runs) |
| 09d | `09d_does_the_human_help` | Does the human step help, and what does it cost? | no (reads the saved Part 7 runs) |
| 09e | `09e_queue_and_failures` | The review queue and what goes wrong | one cell uses the model server |

**The picture.** Airport security. A machine looks at every bag. A person looks only at the bags the machine flags, plus a few random ones to check that the machine is not missing things. Nobody hand-searches every suitcase (too slow), and nobody lets the machine decide alone for the bag it is unsure about (too risky).

Part 7 already ended every image with a verdict word: `pass`, `review` or `escalate`. In Part 7 they were only labels. In this series they become **hand-off words**: who gets to decide next.

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (only 09e calls it)
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
%matplotlib inline
import hitl_eval as ev
from hitl_graph import POLICIES
print("images:", len(ev.TEST), "| policies:", list(POLICIES))

images: 200 | policies: ['none', 'escalate_only', 'all_nonpass', 'all_plus_audit']


## Part 7's verdicts, as they were saved

Two saved Part 7 runs on the same 200 held-out images: `band` (a contrast rule, no model) and `llm` (a language model classifies each box; that run used the local `qwen3-coder`, not the model this notebook set uses, because the run is saved). The notebooks reuse these saved runs instead of calling a model 600 times.

In [2]:
from collections import Counter
for variant in ("band", "llm"):
    res = ev.load_saved(variant)
    print(f"{variant:5s}", dict(Counter(r["verdict"] for r in res)))

band  {'pass': 104, 'review': 93, 'escalate': 3}
llm   {'pass': 109, 'escalate': 36, 'review': 55}


## What is really inside each verdict?

The synthetic dataset has the exact answer for every image, so we can open each verdict class and count what is in it: how many boxes the pipeline **kept**, how many of those were real, and how many real boxes it had **thrown away**. This is the honest picture of what a person will be looking at.

In [3]:
print(f"{'run':5s} {'verdict':9s} {'images':>6s} {'kept boxes':>11s} {'of them real':>13s} {'real boxes thrown away':>23s}")
for variant in ("band", "llm"):
    res = ev.load_saved(variant)
    for verdict in ("pass", "review", "escalate"):
        sub = [r for r in res if r["verdict"] == verdict]
        kept = [(r["image"], k["id"]) for r in sub for k in r["report"]["kept"]]
        kept_real = sum(ev.truth(im)[i] for im, i in kept)
        kept_set = set(kept)
        lost = sum(1 for r in sub for i, real in ev.truth(r["image"]).items() if real and (r["image"], i) not in kept_set)
        print(f"{variant:5s} {verdict:9s} {len(sub):6d} {len(kept):11d} {kept_real:13d} {lost:23d}")

run   verdict   images  kept boxes  of them real  real boxes thrown away


band  pass         104           0             0                       9
band  review        93         143            59                      10
band  escalate       3          17             3                       0
llm   pass         109           0             0                      11
llm   review        55          80            35                       8
llm   escalate      36          66            25                       2


Read the `pass` rows. A `pass` image has **no kept boxes** by definition, so the "real boxes thrown away" column there is the cost of the pipeline's own judgment, and no person will ever see those images under a policy that only looks at `review` and `escalate`.

In `review` and `escalate` images, many kept boxes are false alarms. That is what a person fixes.

## What the reviewer is shown

The first image that makes the graph stop. The graph is the real one from `hitl_graph.py`; the classifier is the rule (`band`) so this runs offline. `payload` is what the reviewer receives: plain JSON, so any form, web page or chat message can show it.

In [4]:
from langgraph.checkpoint.memory import MemorySaver
import hitl_graph as h
g = h.build_hitl("band", "all_nonpass", checkpointer=MemorySaver())
for im in ev.TEST:
    r = h.start(g, im, im)
    if r["status"] == "waiting":
        break
print("first image that needs a person:", im)
print(json.dumps(r["payload"], indent=1)[:1800])

first image that needs a person: test_00002
{
 "image": "test_00002",
 "pipeline_verdict": "review",
 "why_a_human": "2 confident defect(s) survived the rules",
 "candidates": [
  {
   "id": 0,
   "box": [
    98,
    51,
    118,
    65
   ],
   "contrast": 25.67,
   "area_px": 280,
   "edge_distance_px": 51,
   "pipeline": "kept"
  },
  {
   "id": 1,
   "box": [
    161,
    139,
    167,
    162
   ],
   "contrast": 21.04,
   "area_px": 138,
   "edge_distance_px": 57,
   "pipeline": "kept"
  },
  {
   "id": 2,
   "box": [
    169,
    187,
    175,
    194
   ],
   "contrast": 3.91,
   "area_px": 42,
   "edge_distance_px": 30,
   "pipeline": "rejected by classifier"
  }
 ],
 "all_candidate_ids": [
  0,
  1,
  2
 ],
 "question": "approve = the pipeline's kept boxes are exactly the real defects; reject = none is real; edit = give the ids that are real (keep_ids), including boxes the pipeline rejected"
}


In [5]:
from hitl_view import show_review
show_review(r["payload"])

Solid green boxes are the ones the pipeline kept; dashed grey ones are the ones it rejected. The numbers are the candidate ids the reviewer will answer with.

**Try it:** look at the picture and decide which numbered boxes are real voids. You will answer exactly this in notebook 09b, and you can then check yourself against the exact answer.

## The four policies

A policy says which verdicts reach a person. Each skipped class is a risk you accept, so the policies are written down as data, not hidden in `if` statements.

| policy | a person sees | what you accept |
|---|---|---|
| `none` | nothing | the pipeline's `review` verdicts become `fail` on their own |
| `escalate_only` | only `escalate` | confident (but maybe wrong) defects fail without a look |
| `all_nonpass` | `review` and `escalate` | passes are trusted |
| `all_plus_audit` | the above, plus 1 in 10 of the passes | a small sample checks the passes |

In [6]:
for k, p in POLICIES.items():
    print(f"{k:15s} {p.name}")

none            none: the pipeline decides alone
escalate_only   humans see only 'escalate'
all_nonpass     humans see 'review' and 'escalate'
all_plus_audit  'review' + 'escalate' + 1 in 10 of the passes
